# Data Transformation Visualization: Intermediate Activations

Captures real intermediate activations from a trained `exp2b_flash_learned_pool` checkpoint
to produce the 6-stage data transformation strip visualization.

**Stages:**
1. Raw clinical codes (text, one day)
2. After embedding lookup → `[80, 256]`
3. After Learned Attention Pooling → `[1, 256]`
4. All days stacked (after demo injection) → `[200, 256]`
5. After 6 temporal layers → `[200, 256]`
6. Final member embedding → `[256]`

Run on GCP Vertex AI where the trained checkpoint is available.

In [ ]:
import gc
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import google.auth
from google.cloud import bigquery

# Core module — adjust path if running from a different working directory
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'moe'))
from moe_flashattn_4_core import (
    FlashAttentionConfig,
    ClinicalDataset,
    get_experiment_configs,
    _create_model,
    _calculate_model_dimensions,
)

## 1. Configuration

In [ ]:
# ================================================================
# UPDATE these two values before running
# ================================================================
EXP_NAME = 'exp2b_flash_learned_pool'
CHECKPOINT_PATH = (
    'logs/exp_round10_3lobs/exp2b_flash_learned_pool'
    '/checkpoints/checkpoint_best.pt'
)

EMBEDDING_SIZE = 256
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Load Sample Data from BigQuery

In [ ]:
credentials, project = google.auth.default()
client = bigquery.Client()
print(f'BQ project: {project}')

# Fetch members with rich histories (dt_cnt >= 80) — small sample for visualization only
viz_sql = """
SELECT
    individual_id, lob, index_dt,
    gender_cd, age_in_months,
    cd, target, dt_cnt
FROM `edp-prod-storage.edp_ent_sdoheir_cns.a834793_Combined_All_LOB_o3_train_ending`
WHERE dt_cnt >= 80
LIMIT 500
"""
raw_df = client.query(viz_sql).to_dataframe()
print(f'Fetched {len(raw_df)} rows, {raw_df["individual_id"].nunique()} unique members')

# Keep only members with exactly one record (same filtering used during training)
counts = raw_df.groupby('individual_id').size()
member_df = raw_df[raw_df['individual_id'].isin(counts[counts == 1].index)].copy()
print(f'{len(member_df)} single-record members retained')

# Pick the member with the highest dt_cnt for the richest visualization
best_id = member_df.loc[member_df['dt_cnt'].idxmax(), 'individual_id']
sample_df = member_df[member_df['individual_id'] == best_id].reset_index(drop=True)
dt_cnt = int(sample_df['dt_cnt'].iloc[0])
print(f'Selected member {best_id}: {dt_cnt} valid days, LOB={sample_df["lob"].iloc[0]}')

## 3. Build Dataset and Input Tensor

In [ ]:
# ClinicalDataset uses default FlashAttentionConfig vocab sizes
# (len_dy=200, len_cd=80, cd_cnt=75516, target_cd_cnt=6297, embedding_size=256)
config_data = FlashAttentionConfig()
dataset = ClinicalDataset(sample_df, config_data)
sample = dataset[0]   # dict keys: age, gender, lob, codes, dt_cnt, target

age    = sample['age'].unsqueeze(0).to(device)      # [1, 200]
gender = sample['gender'].unsqueeze(0).to(device)   # [1, 200]
lob    = sample['lob'].unsqueeze(0).to(device)       # [1, 200]
codes  = sample['codes'].unsqueeze(0).to(device)    # [1, 200, 80]

# Model input: prepend 3 demographic channels before the code sequence
x = torch.cat([
    age.unsqueeze(-1),     # [1, 200, 1]
    gender.unsqueeze(-1),  # [1, 200, 1]
    lob.unsqueeze(-1),     # [1, 200, 1]
    codes                  # [1, 200, 80]
], dim=-1)                 # → [1, 200, 83]
print(f'Input tensor x: {tuple(x.shape)}')

# Auto-select day_idx as the day with the most active codes (best for Stage 2/3 panels)
active_per_day = (codes[0] > 0).sum(dim=-1).cpu().numpy()   # [200]
day_idx = int(active_per_day[:dt_cnt].argmax())
print(f'Visualization day_idx={day_idx}  ({int(active_per_day[day_idx])} active codes)')

## 4. Load Model

In [ ]:
# Load checkpoint FIRST — read the exact architecture it was trained with
# (avoids nhid/use_learnt_att_pool mismatches from recomputing dims)
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
saved_cfg = ckpt.get('config', {})
print('Checkpoint saved config:', saved_cfg)

# Reconstruct FlashAttentionConfig from the checkpoint's saved fields
from moe_flashattn_4_core import FlashAttentionTransformer
config = FlashAttentionConfig(
    embedding_size=saved_cfg.get('embedding_size', EMBEDDING_SIZE),
    nhid=saved_cfg.get('nhid', 512),
    nhead=saved_cfg.get('nhead', 8),
    nlayers=saved_cfg.get('nlayers', 6),
    dropout=saved_cfg.get('dropout', 0.1),
    use_learnt_att_pool=saved_cfg.get('use_learnt_att_pool', True),
    use_swiglu=saved_cfg.get('use_swiglu', True),
    use_rope=saved_cfg.get('use_rope', True),
    use_flash=saved_cfg.get('use_flash', True),
)
print(f'Config: embedding_size={config.embedding_size}, nhid={config.nhid}, '
      f'nhead={config.nhead}, nlayers={config.nlayers}, '
      f'use_learnt_att_pool={config.use_learnt_att_pool}')

model = FlashAttentionTransformer(config).to(device)

# Strip DataParallelWrapper prefix from keys if present
state_dict = ckpt['model_state_dict']
cleaned = {}
for k, v in state_dict.items():
    k2 = k.replace('module.', '', 1) if k.startswith('module.') else k
    k2 = k2[len('model.'):] if k2.startswith('model.') else k2
    cleaned[k2] = v

try:
    model.load_state_dict(state_dict, strict=True)
    print('Loaded (strict, original keys)')
except RuntimeError:
    try:
        model.load_state_dict(cleaned, strict=True)
        print('Loaded (strict, cleaned keys)')
    except RuntimeError as e:
        model.load_state_dict(cleaned, strict=False)
        print(f'Warning: non-strict load — {e}')

model.eval()
print(f'\nModel ready: embedding_size={ckpt.get("embedding_size")}, '
      f'nlayers={ckpt.get("nlayers")}, type={ckpt.get("model_type")}, '
      f'timestamp={ckpt.get("timestamp")}')
print(f'daily_pooling present: {hasattr(model, "daily_pooling")}')
print(f'temporal_layers: {len(model.temporal_layers)}')

## 5. Register Forward Hooks

In [ ]:
activations = {}

def make_hook(name):
    def hook(module, input, output):
        # FlashAttentionLayer returns (out, attn_weights) — take index 0
        if isinstance(output, tuple):
            activations[name] = output[0].detach().cpu()
        else:
            activations[name] = output.detach().cpu()
    return hook

hooks = []

# Stage 2: embedding lookup [batch, len_dy, len_cd, embedding_size]
hooks.append(model.embedding_cd.register_forward_hook(make_hook('stage2_embeddings')))

# Stage 3: LearnedAttentionPooling → [batch_or_batch*len_dy, len_cd, embedding_size]
if hasattr(model, 'daily_pooling'):
    hooks.append(model.daily_pooling.register_forward_hook(make_hook('stage3_after_lap')))
    print('  daily_pooling hooked')
else:
    print('  WARNING: no daily_pooling found')

# Stage 5: temporal_layers[i] is a ModuleDict — hook the FlashAttentionLayer inside it
# The attention sub-module output is the best proxy for "after temporal block i"
for i, layer_dict in enumerate(model.temporal_layers):
    hooks.append(
        layer_dict['attention'].register_forward_hook(make_hook(f'stage5_layer_{i}'))
    )
    print(f'  temporal_layers[{i}][attention] hooked  ({type(layer_dict["attention"]).__name__})')

print(f'\nTotal hooks registered: {len(hooks)}')

## 6. Forward Pass

In [ ]:
with torch.no_grad():
    output = model(x)   # x: [1, 200, 83]  →  output: [1, 200, target_cd_cnt]

for h in hooks:
    h.remove()

print(f'Output logits: {tuple(output.shape)}')
print('\nCaptured activations:')
for k, v in activations.items():
    print(f'  {k:35s}  shape={tuple(v.shape)}')

## 7. Inspect Activation Statistics

In [ ]:
for key, tensor in activations.items():
    d = tensor.numpy()
    print(f'{key:35s}  shape={str(tuple(d.shape)):25s}  '
          f'min={d.min():.3f}  max={d.max():.3f}  '
          f'mean={d.mean():.3f}  std={d.std():.3f}')

## 8. Main Visualization: 6-Stage Transformation Strip

In [ ]:
# Diagnostic: print what was actually captured
print("Captured activation keys:", list(activations.keys()))
if not any(k.startswith('stage5_layer_') for k in activations):
    raise RuntimeError(
        "No stage5_layer_* activations found.\n"
        "Re-run cells in order: 11 (register hooks) → 13 (forward pass) → this cell."
    )

def to_2d(tensor, day_idx=None):
    """Normalize any activation tensor to 2D numpy [rows, 256].
    Handles shapes: [batch, T, D], [T, D], [batch, D], [D].
    If day_idx is given, returns [1, D] for that day.
    """
    if isinstance(tensor, torch.Tensor):
        tensor = tensor.numpy()
    # Drop batch dim if shape is [1, T, D]
    if tensor.ndim == 3 and tensor.shape[0] == 1:
        tensor = tensor[0]             # → [T, D]
    # Promote 1D [D] to [1, D]
    if tensor.ndim == 1:
        tensor = tensor[np.newaxis, :]
    if day_idx is not None:
        return tensor[day_idx:day_idx+1]   # [1, D]
    return tensor                           # [T, D]


def plot_heatmap(ax, data, title, ylabel='', xlabel='256 dims', cmap='RdBu_r'):
    if isinstance(data, torch.Tensor):
        data = data.numpy()
    # Ensure 2D
    while data.ndim > 2:
        data = data[0]
    if data.ndim == 1:
        data = data[np.newaxis, :]
    vmax = max(abs(data.min()), abs(data.max()))
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax) if vmax > 0 else None
    ax.imshow(data, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xlabel(xlabel, fontsize=7)
    ax.tick_params(labelsize=6)


fig = plt.figure(figsize=(26, 10))
gs = gridspec.GridSpec(2, 6, height_ratios=[1, 4], hspace=0.35, wspace=0.35)

# Stage 2: Embedding lookup [80, 256] for the chosen day
ax1 = fig.add_subplot(gs[:, 0])
emb_day = to_2d(activations['stage2_embeddings'])   # [batch*len_dy*len_cd, D] or [batch, len_dy, len_cd, D]
# embedding_cd output is [batch, len_dy, len_cd, D] — take batch=0, day=day_idx
emb_raw = activations['stage2_embeddings']
if isinstance(emb_raw, torch.Tensor):
    emb_raw = emb_raw.numpy()
if emb_raw.ndim == 4:                       # [batch, len_dy, len_cd, D]
    emb_day = emb_raw[0, day_idx]           # [len_cd, D]
elif emb_raw.ndim == 3:                     # [batch*len_dy, len_cd, D] — can't recover day_idx
    emb_day = emb_raw[day_idx]              # use day_idx as row index
else:
    emb_day = emb_raw
plot_heatmap(ax1, emb_day,
             f'Stage 2: Embedding Lookup\n[80, 256]  Day {day_idx}',
             ylabel='80 code slots')

# Stage 3: After LAP [1, 256] for the same day
ax2 = fig.add_subplot(gs[0, 1])
lap_day = to_2d(activations['stage3_after_lap'], day_idx=day_idx)   # [1, 256]
plot_heatmap(ax2, lap_day, 'Stage 3: After LAP\n[1, 256]', ylabel='1 day')

# Stage 4: All 200 days (pre-temporal encoder)
ax3 = fig.add_subplot(gs[:, 2])
stage4 = to_2d(activations.get('stage4_after_demo', activations['stage3_after_lap']))  # [200, 256]
plot_heatmap(ax3, stage4,
             'Stage 4: All Days (pre-temporal)\n[200, 256]',
             ylabel='200 days')

# Stage 5a: After temporal layer 0
ax4 = fig.add_subplot(gs[:, 3])
plot_heatmap(ax4, to_2d(activations['stage5_layer_0']),
             'Stage 5a: After Temporal Layer 0\n[200, 256]', ylabel='200 days')

# Stage 5b: After the final temporal layer
n_layers = len([k for k in activations if k.startswith('stage5_layer_')])
ax5 = fig.add_subplot(gs[:, 4])
plot_heatmap(ax5, to_2d(activations[f'stage5_layer_{n_layers-1}']),
             f'Stage 5b: After Temporal Layer {n_layers-1}\n[200, 256]', ylabel='200 days')

# Stage 6: Final member embedding — last valid day
ax6 = fig.add_subplot(gs[0, 5])
last_day = dt_cnt - 1
final_emb = to_2d(activations[f'stage5_layer_{n_layers-1}'], day_idx=last_day)  # [1, 256]
plot_heatmap(ax6, final_emb,
             f'Stage 6: Member Embedding\n[1, 256]  Day {last_day}',
             ylabel='')

plt.suptitle(
    'Data Transformation Through Clinical TE Architecture\n(exp2b_flash_learned_pool)',
    fontsize=13, fontweight='bold', y=1.01
)
out_path = 'data_transformation_strip.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

## 9. Stage 4 vs Stage 5 Side-by-Side (Key Contrast Slide)

This is the most impactful comparison: Stage 4 rows look independent/noisy;
Stage 5 rows show vertical stripes and temporal continuity — what 6 causal attention layers learned.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

pre_np  = to_2d(activations.get('stage4_after_demo', activations['stage3_after_lap']))
post_np = to_2d(activations[f'stage5_layer_{n_layers-1}'])

vmax = max(abs(pre_np).max(), abs(post_np).max())
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

axes[0].imshow(pre_np, aspect='auto', cmap='RdBu_r', norm=norm, interpolation='nearest')
axes[0].set_title(
    'Stage 4: Pre-Temporal Encoder\n[200, 256] — rows are independent',
    fontsize=11, fontweight='bold'
)
axes[0].set_ylabel('200 days (rows)', fontsize=10)
axes[0].set_xlabel('256 embedding dims', fontsize=10)

im = axes[1].imshow(post_np, aspect='auto', cmap='RdBu_r', norm=norm, interpolation='nearest')
axes[1].set_title(
    f'Stage 5: After {n_layers} Temporal Layers\n[200, 256] — vertical stripes, temporal continuity',
    fontsize=11, fontweight='bold'
)
axes[1].set_xlabel('256 embedding dims', fontsize=10)

fig.colorbar(im, ax=axes, shrink=0.6, label='Activation value')
plt.suptitle('The Key Contrast: What Causal Attention Layers Learn',
             fontsize=13, fontweight='bold', y=1.02)

out_path2 = 'stage4_vs_stage5_contrast.png'
plt.savefig(out_path2, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path2}')

## 10. Attention Weights for Stage 3 (LAP)

Shows which code slots the model weighted most on `day_idx`.
Requires `LearnedAttentionPooling.forward()` to save `self.last_attn_weights`.

In [ ]:
if hasattr(model, 'daily_pooling') and hasattr(model.daily_pooling, 'last_attn_weights'):
    attn = model.daily_pooling.last_attn_weights   # [batch, len_dy, len_cd] or [batch, len_dy, 1, len_cd]
    if attn.dim() == 4:
        attn = attn.squeeze(2)
    attn_day = attn[0, day_idx].cpu().numpy()   # [80]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(range(len(attn_day)), attn_day[::-1], color='steelblue')
    ax.set_xlabel('Attention weight', fontsize=10)
    ax.set_ylabel('Code slot (0=top)', fontsize=10)
    ax.set_title(f'LAP Attention Weights — Day {day_idx}\n'
                 f'(higher weight = more clinically relevant to model)',
                 fontsize=11)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('lap_attention_weights.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved: lap_attention_weights.png')
else:
    print(
        'Attention weights not available.\n'
        'To enable: add `self.last_attn_weights = attn_weights` inside '
        'LearnedAttentionPooling.forward() before returning.'
    )

## 11. Layer-by-Layer Evolution

Shows how the temporal representation builds depth across all 6 layers.

In [ ]:
layer_keys = sorted([k for k in activations if k.startswith('stage5_layer_')])
print(f'Temporal layers captured: {len(layer_keys)}')

fig, axes = plt.subplots(1, len(layer_keys), figsize=(4 * len(layer_keys), 8), sharey=True)
if len(layer_keys) == 1:
    axes = [axes]

all_data = [to_2d(activations[k]) for k in layer_keys]
vmax = max(abs(d).max() for d in all_data)
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

for i, (ax, d) in enumerate(zip(axes, all_data)):
    ax.imshow(d, aspect='auto', cmap='RdBu_r', norm=norm, interpolation='nearest')
    ax.set_title(f'Layer {i}', fontsize=10, fontweight='bold')
    ax.set_xlabel('256 dims', fontsize=8)
    if i == 0:
        ax.set_ylabel('200 days', fontsize=9)
    ax.tick_params(labelsize=6)

plt.suptitle('Representation After Each Temporal Layer (left=shallow → right=deep)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('temporal_layer_evolution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: temporal_layer_evolution.png')